# Myra AI - Level 4 Fine-Tuning with Hendrycks MATH & LLaMA-Factory 🐻

This notebook runs **Supervised Fine-Tuning (SFT) with LoRA** to train open-source models (`Qwen/Qwen2.5-Math-1.5B-Instruct`, `Qwen/Qwen2.5-Math-7B-Instruct`, or `meta-llama/Llama-3.1-8B-Instruct`) to solve high-school competition math in **Myra AI's friendly, step-by-step pedagogical persona**.

### Requirements
- Google Colab runtime: **T4 GPU** (Free) or **A100**.

In [ ]:
# 1. Verify GPU Hardware
!nvidia-smi

In [ ]:
# 2. Clone LLaMA-Factory and install dependencies
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
%cd LlamaFactory
!pip install -e ".[torch,metrics,bitsandbytes]"
!pip install pyarrow pandas

In [ ]:
# 3. Download Hendrycks MATH and format into Myra Dialogue Persona
import urllib.request, json, os, pandas as pd

os.makedirs('data', exist_ok=True)
parquet_url = 'https://huggingface.co/datasets/qwedsacf/competition_math/resolve/main/data/train-00000-of-00001-7320a6f3aba8ebd2.parquet'
urllib.request.urlretrieve(parquet_url, 'data/train.parquet')
print('MATH dataset downloaded!')

df = pd.read_parquet('data/train.parquet')
print(f'Total problems: {len(df)}')

def extract_boxed(sol):
    idx = sol.rfind('\\boxed{')
    if idx == -1: return ''
    start = idx + len('\\boxed{')
    depth, curr = 1, start
    while curr < len(sol) and depth > 0:
        if sol[curr] == '{': depth += 1
        elif sol[curr] == '}': depth -= 1
        curr += 1
    return sol[start:curr-1].strip()

system_prompt = (
    "You are Myra, the friendly AI study buddy from EdWorld Co. Your mascot is a teddy bear in a hoodie. "
    "Explain math step by step in simple English. Always end with: 'Does this make sense? Want a practice question?'"
)

# Sample 2,500 balanced problems
dialogues = []
for _, row in df.sample(n=2500, random_state=42).iterrows():
    boxed = extract_boxed(row['solution'])
    intro = f"The final answer is ${boxed}$." if boxed else "Here is the step-by-step solution:"
    myra_reply = f"{intro}\n\nLet us walk through the steps together:\n\n{row['solution'].strip()}\n\nDoes this make sense? Want a practice question?"
    dialogues.append({
        'conversations': [
            {'from': 'human', 'value': row['problem'].strip()},
            {'from': 'gpt', 'value': myra_reply}
        ],
        'system': system_prompt
    })

with open('data/myra_math_sft.json', 'w', encoding='utf-8') as f:
    json.dump(dialogues, f, indent=2, ensure_ascii=False)

# Register in dataset_info.json
with open('data/dataset_info.json', 'r') as f: info = json.load(f)
info['myra_math'] = {'file_name': 'myra_math_sft.json', 'formatting': 'sharegpt', 'columns': {'messages': 'conversations', 'system': 'system'}}
with open('data/dataset_info.json', 'w') as f: json.dump(info, f, indent=2)
print('Registered myra_math dataset in LLaMA-Factory!')

In [ ]:
# 4. Launch LoRA Supervised Fine-Tuning
!llamafactory-cli train \
    --stage sft \
    --do_train true \
    --model_name_or_path Qwen/Qwen2.5-Math-1.5B-Instruct \
    --dataset myra_math \
    --template qwen \
    --finetuning_type lora \
    --lora_target all \
    --output_dir saves/myra-math-lora \
    --overwrite_output_dir true \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --save_steps 200 \
    --learning_rate 1e-4 \
    --num_train_epochs 3.0 \
    --fp16 true

In [ ]:
# 5. Test Inference with Trained Myra LoRA Adapter
!llamafactory-cli chat \
    --model_name_or_path Qwen/Qwen2.5-Math-1.5B-Instruct \
    --adapter_name_or_path saves/myra-math-lora \
    --template qwen

In [ ]:
# 6. Export Merged Model for Production Serving
!llamafactory-cli export \
    --model_name_or_path Qwen/Qwen2.5-Math-1.5B-Instruct \
    --adapter_name_or_path saves/myra-math-lora \
    --template qwen \
    --export_dir exports/myra-math-model \
    --export_size 2 \
    --export_device cpu